# Chapter 3 Practical 02: Cosine, Pearson, and Jaccard Similarity

Learning objectives:
- Compute user-user similarity on co-rated items.
- Compare cosine and Pearson similarity for explicit ratings.
- Use Jaccard similarity for binary interactions.
- Apply minimum-overlap and shrinkage to reduce noisy similarities.

Slide connection: similarity measures, cosine equation, Pearson equation, and sparse overlap.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("chapter_03_collaborative_filtering/data")

ratings = pd.read_csv(DATA_DIR / "ratings_chapter3.csv")
movies = pd.read_csv(DATA_DIR / "movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


In [ ]:
def common_ratings(matrix, user_a, user_b):
    pair = matrix.loc[[user_a, user_b]].dropna(axis=1)
    return pair.loc[user_a], pair.loc[user_b]

def cosine_on_overlap(matrix, user_a, user_b):
    a, b = common_ratings(matrix, user_a, user_b)
    if len(a) == 0:
        return np.nan
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return np.nan if denom == 0 else float(np.dot(a, b) / denom)

def pearson_on_overlap(matrix, user_a, user_b):
    a, b = common_ratings(matrix, user_a, user_b)
    if len(a) < 2:
        return np.nan
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def jaccard_liked(matrix, user_a, user_b, threshold=5):
    liked_a = set(matrix.columns[matrix.loc[user_a] >= threshold])
    liked_b = set(matrix.columns[matrix.loc[user_b] >= threshold])
    if not liked_a and not liked_b:
        return np.nan
    return len(liked_a & liked_b) / len(liked_a | liked_b)


In [ ]:
rows = []
target = "Karen"
for other in rating_matrix.index.drop(target):
    overlap_count = rating_matrix.loc[[target, other]].notna().all(axis=0).sum()
    rows.append({
        "target_user": target,
        "other_user": other,
        "co_rated_items": int(overlap_count),
        "cosine": cosine_on_overlap(rating_matrix, target, other),
        "pearson": pearson_on_overlap(rating_matrix, target, other),
        "jaccard_liked": jaccard_liked(rating_matrix, target, other),
    })

similarities = pd.DataFrame(rows).sort_values("pearson", ascending=False)
similarities.round(3)


Pearson removes each user's average rating behavior. This matters when one user rates generously and another rates strictly.


In [ ]:
def similarity_matrix(metric):
    users = rating_matrix.index
    sim = pd.DataFrame(index=users, columns=users, dtype=float)
    for u in users:
        for v in users:
            sim.loc[u, v] = metric(rating_matrix, u, v) if u != v else 1.0
    return sim

pearson_sim = similarity_matrix(pearson_on_overlap)
pearson_sim.round(2)


Shrinkage discounts similarities based on very small overlap.


In [ ]:
def shrink_similarity(similarity, overlap_count, alpha=3):
    if pd.isna(similarity):
        return np.nan
    return similarity * overlap_count / (overlap_count + alpha)

similarities["pearson_shrunk"] = similarities.apply(
    lambda row: shrink_similarity(row["pearson"], row["co_rated_items"], alpha=3),
    axis=1,
)
similarities.sort_values("pearson_shrunk", ascending=False).round(3)


Exercises:
1. Change the liked threshold for Jaccard from 5 to 6.
2. Increase the shrinkage alpha. Which neighbors lose influence?
